# Build optical LUTs

Install the repository into this notebook's Python environment first:
`python -m pip install -e /path/to/SizeDistMerge`.
The Python files live directly in `src/`; the public import remains `sizedistmerge`.
No private data or absolute machine-specific paths are included.

This example uses the shared optical-geometry interface for POPS, UHSAS
and PCASP. Building all three full-resolution tables is expensive. It is disabled
by default. New tables go to a separate output directory; this notebook never
overwrites the repository LUTs or starts a campaign merge.

The diameter grid is a calculation range, not the instrument measurement range.
The 100 response groups used during conversion are separate from these 1,000
diameter samples.

In [ ]:
from pathlib import Path
import numpy as np
import zarr
from sizedistmerge import optical_diameter as od
from sizedistmerge.optical_geometry import load_optical_setup
from sizedistmerge.optical_lut import build_setup_sigma_lut

RUN_BUILD = False
OUTPUT_DIR = Path('lut_build_example')
WORKERS = 2
N_RANGE = (1.30, 1.80, 0.0005)
# 200 values: zero plus a logarithmic positive grid. No calibration is assumed.
K_VALUES = np.r_[0., np.geomspace(1e-4, 0.8, 199)]
SPECS = {
    # Built-in files live in opc_setups/. A custom .toml path also works.
    'pops': (load_optical_setup('pops'), 'Collection', (60., 6000., 1000)),
    'uhsas': (load_optical_setup('uhsas'), 'Collection 1', (30., 6000., 1000)),
    'pcasp': (load_optical_setup('pcasp'), 'Collection', (60., 6000., 1000)),
}

In [ ]:
if RUN_BUILD:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for name, (setup, channel, diameter_range) in SPECS.items():
        destination = OUTPUT_DIR / f'{name}.zarr'
        if destination.exists():
            raise FileExistsError(f'Refusing to overwrite {destination}')
        build_setup_sigma_lut(
            str(destination), setup, channel=channel,
            D_range=diameter_range, n_range=N_RANGE, k_values=K_VALUES,
            chunks=(128, 64, 1), jobs_per_k=WORKERS,
            parallel_backend='threads')
        saved = zarr.open_group(destination, mode='r')
        assert saved.attrs['build_complete'] is True
        # Small numerical spot check; this is not instrument calibration validation.
        di, ni, ki = 100, 400, 0
        diameter = float(saved['coords/D_nm'][di])
        ri = complex(float(saved['coords/n'][ni]), float(saved['coords/k'][ki]))
        direct = od.setup_csca([diameter], ri, setup)[channel][0]
        np.testing.assert_allclose(saved['sigma_col'][di, ni, ki], direct, rtol=1e-6)
        print('Completed:', destination)
else:
    print('Build disabled. Review settings before setting RUN_BUILD=True.')